# Argus prototype data 4: stellar flares

Minipixes exist only where the pipeline kept them in this reduction (around stars to G ≈ 16, and for selected targets and transients in Array production). In this sample data, only static sources are extracted as minipix. 
Anything else, such as a faint M dwarf caught flaring, is still in the
archived segment frames. The sample holds four flares found by the
prototype's live difference-imaging pipeline, each as a stack of 32 × 32
cutouts from the segment frames in both bands, around the peak.

Those cutouts carry no records of their own, but the photocal model belongs
to the frame, so a nearby minipix's records calibrate them, matched by frame
key. Each flare comes with the five nearest active minipixes over the same
time window. Notebooks 2 and 3 cover minipixes and the photometry.

## Parameters

Change these inputs first; leave the calculation cells intact.

Pick a flare with `FLARE_ID`. The first is the clearest; the others are
listed with what to look for.

| Gaia DR3 | G | BP − RP | Peak (UTC) | Notes |
|---|---|---|---|---|
| 313993690233192704 | 17.0 | 2.85 | 2026-10-13 06:40:10 | a fast rise, 40× in β |
| 316701375055391872 | | | 2026-10-17 05:31:42 | a slow rise, still near peak when the pointing moved on |
| 241255975238756224 | | | 2026-10-19 08:13:18 | about 30 minutes of decay |
| 239601141519300096 | | | 2026-10-20 09:49:32 | 50 quiet minutes before it; a G = 17.7 star 4.9″ away dilutes the amplitude |

In [ ]:
FLARE_ID = "313993690233192704"
BP_RP = {"313993690233192704": 2.85}  # Gaia DR3; the others use the zeropoint's pivot color

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices
GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# The staged sample. Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

## 1. The flare's cutouts

In [ ]:
import json

d = SAMPLE / "flares" / FLARE_ID
flare = json.load(open(d / "flare.json"))
f_ra, f_dec = flare["ra"], flare["dec"]
t_peak = times.to_epoch(flare["peak_utc"])
f_stacks = {band: argex.read(d / f"stack_{band}.fits") for band in ("beta", "rho")}
for band, st in f_stacks.items():
    print(f"{band}: {len(st)} frames, {times.to_iso(st.epochs[0])[11:19]}"
          f" to {times.to_iso(st.epochs[-1])[11:19]} UTC")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 3.8))
for row, (band, st) in zip(axes, f_stacks.items()):
    k_peak = int(np.argmin(np.abs(st.epochs - t_peak)))
    norm = ImageNormalize(st.data[k_peak], stretch=AsinhStretch(0.02))
    for ax, k in zip(row, range(k_peak - 3, k_peak + 3)):
        if 0 <= k < len(st):
            ax.imshow(st.data[k], norm=norm, cmap="gray")
            ax.set_title(f"{GREEK[band]} {(st.epochs[k] - t_peak) / 60:+.0f} min", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
plt.show()

These cutouts have no DQ records, so ruling out an artifact comes down to
the images. A cosmic ray or a satellite trail hits one camera's frame. A
flare appears in both cameras in the same exposure, has the shape of a star,
and fades over several frames.

## 2. Borrowing zeropoints

The nearest active minipixes were exposed in the same frames. Evaluating
each one's photocal model at the flare star's segment position and color
gives several estimates of each frame's zeropoint there. They agree to a few
mmag: the model's gradient and color terms are small.

In [ ]:
bp_rp = BP_RP.get(FLARE_ID)
f_gr = None if bp_rp is None else (bp_rp - 0.355) / 1.198
zps = {}  # frame key -> the zeropoint at the flare star, from each minipix
for path in sorted((d / "minipix").glob("*.fits")):
    ref = argex.read(path)
    for frame, z in zip(ref.frames, ref.zeropoint(flare["x"] - ref.x0, flare["y"] - ref.y0, gr=f_gr)):
        zps.setdefault(frame.frame_key, []).append(z)
spread = np.nanmedian([np.nanmax(z) - np.nanmin(z) for z in zps.values()])
print(f"{len(zps)} frames from {len(flare['minipix'])} minipixes;"
      f" they agree to {spread * 1000:.1f} mmag (median range)")

## 3. The flare's light curve

The same photutils photometry as in notebook 3, at the zeropoints' 4.05″
radius, with the borrowed zeropoints. There are no DQ records to pass.

In [ ]:
from astropy.stats import SigmaClip
from astropy.table import Table
from photutils.aperture import ApertureStats, CircularAnnulus, CircularAperture, aperture_photometry

EXPTIME = 60.0                                # s, every frame
SEGMENT_PIXEL, DETECTOR_PIXEL = 0.879, 1.012  # arcsec
R = 4 * DETECTOR_PIXEL / SEGMENT_PIXEL        # the zeropoints' 4.05" aperture, in segment pixels
ANNULUS = (R + 3, 15.5)                       # what fits in a 32-pixel cutout


def aperture_flux(cube, x, y, radius=R, annulus=ANNULUS):
    # Background-subtracted flux (e-/s) and its error in each plane of a cube,
    # with the background the sigma-clipped median of the annulus.
    aperture = CircularAperture((x, y), r=radius)
    ring = CircularAnnulus((x, y), r_in=annulus[0], r_out=annulus[1])
    clip = SigmaClip(sigma=3.0)
    flux, err = np.full(len(cube), np.nan), np.full(len(cube), np.nan)
    for k, plane in enumerate(np.asarray(cube, np.float64)):
        sky_ = ApertureStats(plane, ring, sigma_clip=clip)
        phot = aperture_photometry(plane - sky_.median, aperture,
                                   error=np.full(plane.shape, sky_.std), method="exact")
        flux[k] = phot["aperture_sum"][0]
        # the sky noise from the annulus, plus the source's photon noise
        err[k] = np.hypot(phot["aperture_sum_err"][0], np.sqrt(max(flux[k], 0) / EXPTIME))
    return flux, err


def light_curve(frames, epochs, flux, err, zp, flags=None):
    # A table of calibrated magnitudes; a frame is good when it is calibrated,
    # has a positive flux and (with DQ flags) none of argex.dq.BAD under the aperture.
    mag = zp - 2.5 * np.log10(flux)
    good = np.isfinite(mag)
    if flags is not None:
        good &= (flags & int(argex.dq.BAD)) == 0
    return Table({
        "epoch": epochs, "mjd": times.to_mjd(epochs),
        "frame_key": [f.frame_key for f in frames],
        "flux": flux, "flux_err": err, "zp": zp,
        "mag": mag, "mag_err": 2.5 / np.log(10) * err / flux,
        "dq": np.zeros(len(flux), np.uint32) if flags is None else flags, "good": good,
    })

In [ ]:
f_lcs = {}
for band, st in f_stacks.items():
    fx, fy = st.radec_to_pixel(f_ra, f_dec)
    zp = np.array([np.nanmedian(zps.get(f.frame_key, np.nan)) for f in st.frames])
    flux, err = aperture_flux(st.data, fx, fy)
    f_lcs[band] = light_curve(st.frames, st.epochs, flux, err, zp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
for band, color in (("beta", "tab:blue"), ("rho", "tab:red")):
    lc = f_lcs[band]
    minutes = (lc["epoch"] - t_peak) / 60
    quiet = np.nanmedian(lc["flux"][minutes < -2])
    g = lc["good"]
    ax1.errorbar(minutes[g], lc["mag"][g], lc["mag_err"][g], fmt="o-", ms=3, color=color, label=GREEK[band])
    ax2.plot(minutes, lc["flux"] / quiet, "o-", ms=3, color=color, label=GREEK[band])
    k = int(np.nanargmax(lc["flux"]))
    print(f"{band}: quiescent {lc['zp'][k] - 2.5 * np.log10(quiet):.2f},"
          f" peak {lc['mag'][k]:.2f} ± {lc['mag_err'][k]:.2f} AB"
          f" at {minutes[k]:+.1f} min: {lc['flux'][k] / quiet:.0f}x the quiescent flux")
ax1.invert_yaxis()
ax1.set(xlabel="minutes from the alert's peak", ylabel="AB mag", title=f"Gaia DR3 {FLARE_ID}")
ax2.set(xlabel="minutes from the alert's peak", ylabel="flux / quiescent", yscale="log")
ax1.legend(); ax2.legend()
plt.show()

For the first flare, most of the rise, a factor of about ten in β, happens
within a single minute. The decay then takes several minutes; the pointing
moved on before the star was quiet again. The contrast is far larger in β
than in ρ. A flare radiates like a hot, roughly 10,000 K, source, bright in
the blue, while the M dwarf itself emits mostly in the red: in quiescence it
is about two magnitudes fainter in β than in ρ.

Now set `FLARE_ID` in the parameters to one of the others and run the notebook again.